# Bronze — ingestão e catalogação das 3 edições do State of Data Brasil

**Objetivo da camada Bronze**: carregar os CSVs brutos (um por edição da pesquisa,
baixados manualmente do Kaggle por exigirem login) o mais próximo possível de
como vieram da fonte — convertendo para Parquet (mais eficiente que CSV pro
resto do pipeline) e corrigindo o TIPO dos valores ausentes (ver célula de
normalização abaixo) — e catalogar cada edição como uma tabela no AWS Glue
Data Catalog, para ficar consultável via Athena.

**Por que 3 tabelas separadas (`tb_bronze_2023_2024`, `tb_bronze_2024_2025`,
`tb_bronze_2025_2026`), e não uma tabela `bronze` única?** Cada edição tem um
schema de colunas **diferente** (quantidade e nomes de colunas mudam a cada ano
da pesquisa). Um Crawler apontando pra um prefixo com os 3 CSVs juntos cria uma
tabela "Frankenstein": a união de todas as colunas das 3 edições, com `NULL` em
toda coluna que simplesmente não existe naquela edição — ou seja, boa parte do
volume de nulo que aparece numa tabela `bronze` única não é dado faltando, é
efeito colateral de forçar 3 questionários diferentes dentro da mesma tabela.
Por isso aqui cada edição vira sua própria tabela, com seu schema nativo, sem
nulo artificial — a unificação de verdade (mesma pergunta = mesma coluna, nas
3 edições) é trabalho da Silver, não da Bronze.

Pré-requisito: os 3 arquivos CSV devem já estar enviados para os prefixos
`RAW_PATHS` no S3 (upload simples, feito fora deste notebook).

In [ ]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5


In [ ]:
%%configure
{
  "--enable-glue-datacatalog": "true"
}


In [ ]:
import sys
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
import pyspark.sql.functions as F
import pyspark.sql.types as T

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)


In [ ]:
# =============================================================================
# CONFIGURACAO — valores conferidos AO VIVO no console AWS do time em
# 2026-10-18 (Glue Data Catalog + S3). Troque so se os nomes mudarem.
#
# Os 3 CSVs brutos ficam juntos, soltos (sem subpasta por edicao), direto em
# s3://tech-challenge-gt26/bronze/ — e com nomes de arquivo que NAO indicam a
# edicao de forma confiavel. Mapeamento confirmado por inspecao de conteudo
# (S3 Select: formato do cabecalho + pergunta "passou por layoff em <ano>"):
#   - "State_of_data_2022.csv"                      -> edicao 2023-2024
#   - "State_of_data_BR_2023_Kaggle - df_survey_2023.csv" -> edicao 2024-2025
#   - "Final Dataset - State of Data 2025-2026 - Kaggle.csv" -> edicao 2025-2026
#
# Os nomes de tabela abaixo (tb_bronze_*, tb_silver_*, tb_gold_*) usam um
# prefixo "tb_" de proposito para conviver, dentro do MESMO database
# db_techchallenge_gt_fiap, com as tabelas "bronze"/"silver"/"gold" que ja
# existem — nada aqui sobrescreve ou apaga o que ja foi feito.
# =============================================================================
BUCKET = "s3://tech-challenge-gt26"

RAW_PATHS = {
    "2023-2024": f"{BUCKET}/bronze/State_of_data_2022.csv",
    "2024-2025": f"{BUCKET}/bronze/State_of_data_BR_2023_Kaggle - df_survey_2023.csv",
    "2025-2026": f"{BUCKET}/bronze/Final Dataset - State of Data 2025-2026 - Kaggle.csv",
}

# Prefixo NOVO e distinto do "bronze/" (lowercase) onde os CSVs brutos ja
# estao — evita qualquer risco de sobrescrever a fonte original.
BRONZE_PATH = f"{BUCKET}/bronze_parquet/"
SILVER_PATH = f"{BUCKET}/Silver/"
GOLD_PATH = f"{BUCKET}/Gold/"

DB_BRONZE = "db_techchallenge_gt_fiap"
DB_SILVER = "db_techchallenge_gt_fiap"
DB_GOLD = "db_techchallenge_gt_fiap"

for db in (DB_BRONZE, DB_SILVER, DB_GOLD):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")


## Leitura e normalização do tipo dos valores ausentes, por edição

In [ ]:
# O encoding de origem é UTF-8 com alguns raros bytes corrompidos em campos de
# texto livre (identificado na validação local) — "PERMISSIVE" + multiLine
# evitam que uma linha malformada derrube a leitura inteira do arquivo.
EDICOES = ["2023-2024", "2024-2025", "2025-2026"]

bronze_dfs = {}
for edicao in EDICOES:
    df = (
        spark.read
        .option("header", "true")
        .option("encoding", "UTF-8")
        .option("multiLine", "true")
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .csv(RAW_PATHS[edicao])
    )
    bronze_dfs[edicao] = df
    print(f"{edicao}: {df.count():,} linhas, {len(df.columns)} colunas")


## Renomeação para nomes de coluna compatíveis com o Hive Metastore

O cabeçalho original da edição 2023-2024 vem serializado como tupla Python em
texto (ex: `"('P1_b ', 'Genero')"`) — com vírgula, aspas e parênteses dentro do
nome. O Glue Data Catalog usa o Hive Metastore por trás, que **não aceita
vírgula (nem vários outros caracteres) em nome de coluna** — tentar catalogar
assim gera `AnalysisException: ... column whose name contains commas`.
Isso explica, inclusive, por que o Crawler do time também não preservou os
nomes originais: nenhuma ferramenta compatível com Hive consegue.

A célula abaixo sanitiza os nomes **por posição**, sem depender do significado
de cada coluna — a Silver já seleciona campos pelo índice posicional
(`col_by_idx`), então o nome exato aqui não importa para o resto do pipeline,
só precisa ser válido no Hive e consistente dentro de cada edição.

In [ ]:
import re

def nomes_seguros(colunas):
    nomes, vistos = [], {}
    for i, c in enumerate(colunas):
        nome = re.sub(r"[^0-9a-zA-Z_]+", "_", c).strip("_")[:120].lower()
        if not nome or nome[0].isdigit():
            nome = f"c_{nome}" if nome else f"col{i}"
        if nome in vistos:
            vistos[nome] += 1
            nome = f"{nome}_{vistos[nome]}"
        else:
            vistos[nome] = 0
        nomes.append(nome)
    return nomes

for edicao, df in bronze_dfs.items():
    bronze_dfs[edicao] = df.toDF(*nomes_seguros(df.columns))
print("colunas renomeadas pra nomes compativeis com o Hive Metastore (mantendo a ordem posicional)")


O CSV original representa "não respondi"/"não se aplica" como **string vazia**
(`""`), não como nulo de verdade — como texto, uma célula vazia parece igual a
um nulo, mas pra um motor de consulta (Spark, Athena) os dois são tratados de
forma diferente (`COUNT`, `IS NULL`, `GROUP BY` etc. tratam `""` como um valor
válido, não como ausência de valor, o que distorce qualquer contagem). A célula
abaixo corrige isso de forma genérica, sem precisar saber o que cada coluna
significa: tira espaço em branco (`"   "` também vira nulo) e string vazia
passa a ser `NULL` de verdade — aplicado a **todas** as colunas, já que a
Bronze ainda não sabe o significado de cada uma (isso só é decidido na Silver).
Nada além de tipo muda aqui: nenhuma linha é removida, nenhum valor
não-vazio é alterado.

In [ ]:
def normalizar_nulos(df):
    # Varios cabecalhos da edicao 2023-2024 vem serializados como tupla Python
    # em texto (ex: "('P3_c_1 ', 'Pensar na visao de longo prazo...')"), com
    # ponto/virgula/aspas dentro do nome. Tanto F.col(nome) quanto df[nome]
    # tentam interpretar nomes assim como referencia qualificada/aninhada e
    # quebram com AnalysisException — a unica forma segura e envolver o nome
    # inteiro em crase (`...`), que forca o Spark a trata-lo como um unico
    # identificador literal, sem nenhum parsing de ponto/virgula/aspas.
    #
    # Em vez de 399 .withColumn() encadeados (um plano Spark profundamente
    # aninhado, lento de analisar — e a causa da sessao travando antes), um
    # unico .select() com todas as colunas recalculadas de uma vez: mais
    # rapido e evita reavaliar a cadeia inteira a cada coluna.
    colunas = [
        F.when(F.trim(F.col(f"`{c}`")) == "", None).otherwise(F.trim(F.col(f"`{c}`"))).alias(c)
        for c in df.columns
    ]
    return df.select(*colunas)

bronze_dfs = {edicao: normalizar_nulos(df) for edicao, df in bronze_dfs.items()}


## Diagnóstico: quanto cada edição tem de valor ausente

Mostra, por edição, o percentual médio de células nulas e as 5 colunas com mais
nulo — serve pra confirmar que o volume alto de nulo é **esperado** (a pesquisa
usa lógica de "pule para": quem não usa cloud, por exemplo, não responde as
perguntas sobre qual cloud usa) e não um efeito colateral de bug de leitura.

In [ ]:
for edicao, df in bronze_dfs.items():
    total = df.count()
    linha = df.select([
        F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(f"c{i}")
        for i, c in enumerate(df.columns)
    ]).collect()[0]
    pct_por_coluna = [
        (df.columns[i], round(linha[f"c{i}"] / total * 100, 1))
        for i in range(len(df.columns))
    ]
    media = sum(p for _, p in pct_por_coluna) / len(pct_por_coluna)
    piores = sorted(pct_por_coluna, key=lambda t: t[1], reverse=True)[:5]
    print(f"{edicao}: {media:.1f}% de nulo em média por coluna, de {total:,} linhas")
    print(f"  5 colunas com mais nulo: {piores}")


## Gravação em Parquet e catalogação de cada edição

In [ ]:
for edicao, df in bronze_dfs.items():
    tabela = "tb_bronze_" + edicao.replace("-", "_")
    caminho = BRONZE_PATH + edicao.replace("-", "_") + "/"
    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("path", caminho)
        .saveAsTable(f"{DB_BRONZE}.{tabela}")
    )
    print(f"gravado e catalogado: {DB_BRONZE}.{tabela}  ->  {caminho}")


## Verificação

Confirma que as 3 tabelas Bronze foram catalogadas corretamente e podem ser
consultadas via Spark SQL (e, depois de publicadas, via Athena).

In [ ]:
spark.sql(f"SHOW TABLES IN {DB_BRONZE}").show(truncate=False)
